In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from ingestion.data_loader import DataLoader
from preprocessing.preprocessing import Preprocessor
from features.feature_builder import FeatureBuilder

from analytics.clustering import PeerClusterer
from analytics.benchmarking import PercentileBenchmark
from analytics.ranking import Ranker

In [16]:
# 1. Load
df = DataLoader().load()

In [5]:
df.head()

,entity_id,year,operator_id,type,size,age,km,fuel,co2
0,0,2018,C-015,long-haul,21.680127,4,91468.944340,35.421193,93.511950
1,0,2019,C-015,long-haul,21.680127,5,153835.212095,66.257263,174.919175
2,0,2020,C-015,long-haul,21.680127,6,149495.965275,65.903525,173.985307
3,0,2021,C-015,long-haul,21.680127,7,176381.939246,87.909431,232.080898
4,0,2022,C-015,long-haul,21.680127,8,151980.920927,80.235483,211.821676


In [6]:
# 2. Preprocess
df = Preprocessor().transform(df)

In [7]:
df.head()

,entity_id,year,operator_id,type,size,age,km,fuel,co2,fuel_per_km,co2_per_km
0,0,2018,C-015,long-haul,21.680127,4,91468.944340,35.421193,93.511950,0.000387,0.001022
1,0,2019,C-015,long-haul,21.680127,5,153835.212095,66.257263,174.919175,0.000431,0.001137
2,0,2020,C-015,long-haul,21.680127,6,149495.965275,65.903525,173.985307,0.000441,0.001164
3,0,2021,C-015,long-haul,21.680127,7,176381.939246,87.909431,232.080898,0.000498,0.001316
4,0,2022,C-015,long-haul,21.680127,8,151980.920927,80.235483,211.821676,0.000528,0.001394


In [8]:
# 3. Features
df = FeatureBuilder().build(df)

In [9]:
df.head()

,entity_id,year,operator_id,type,size,age,km,fuel,co2,fuel_per_km,co2_per_km,size_bucket,age_bucket,co2_per_km_lag1
0,0,2018,C-015,long-haul,21.680127,4,91468.944340,35.421193,93.511950,0.000387,0.001022,0,1,NaN
1,0,2019,C-015,long-haul,21.680127,5,153835.212095,66.257263,174.919175,0.000431,0.001137,0,1,0.001022
2,0,2020,C-015,long-haul,21.680127,6,149495.965275,65.903525,173.985307,0.000441,0.001164,0,1,0.001137
3,0,2021,C-015,long-haul,21.680127,7,176381.939246,87.909431,232.080898,0.000498,0.001316,0,1,0.001164
4,0,2022,C-015,long-haul,21.680127,8,151980.920927,80.235483,211.821676,0.000528,0.001394,0,2,0.001316


In [10]:
# 4. Clustering
clusterer = PeerClusterer(n_clusters=6)
clusterer.fit(df)
df = clusterer.transform(df)

In [11]:
df.head()

,entity_id,year,operator_id,type,size,age,km,fuel,co2,fuel_per_km,co2_per_km,size_bucket,age_bucket,co2_per_km_lag1,cluster
0,0,2018,C-015,long-haul,21.680127,4,91468.944340,35.421193,93.511950,0.000387,0.001022,0,1,NaN,0
1,0,2019,C-015,long-haul,21.680127,5,153835.212095,66.257263,174.919175,0.000431,0.001137,0,1,0.001022,0
2,0,2020,C-015,long-haul,21.680127,6,149495.965275,65.903525,173.985307,0.000441,0.001164,0,1,0.001137,4
3,0,2021,C-015,long-haul,21.680127,7,176381.939246,87.909431,232.080898,0.000498,0.001316,0,1,0.001164,4
4,0,2022,C-015,long-haul,21.680127,8,151980.920927,80.235483,211.821676,0.000528,0.001394,0,2,0.001316,4


In [12]:
# 5. Benchmarking
benchmark = PercentileBenchmark(use_clustering=False)
df = benchmark.compute(df)

In [13]:
df.head()

,entity_id,year,operator_id,type,size,age,km,fuel,co2,fuel_per_km,co2_per_km,size_bucket,age_bucket,co2_per_km_lag1,cluster,percentile_co2,zscore_co2
0,0,2018,C-015,long-haul,21.680127,4,91468.944340,35.421193,93.511950,0.000387,0.001022,0,1,NaN,0,0.500000,-0.772947
1,0,2019,C-015,long-haul,21.680127,5,153835.212095,66.257263,174.919175,0.000431,0.001137,0,1,0.001022,0,0.916667,0.789709
2,0,2020,C-015,long-haul,21.680127,6,149495.965275,65.903525,173.985307,0.000441,0.001164,0,1,0.001137,4,0.933333,0.849900
3,0,2021,C-015,long-haul,21.680127,7,176381.939246,87.909431,232.080898,0.000498,0.001316,0,1,0.001164,4,1.000000,1.841436
4,0,2022,C-015,long-haul,21.680127,8,151980.920927,80.235483,211.821676,0.000528,0.001394,0,2,0.001316,4,0.833333,0.947931


In [14]:
# 6. Ranking
df = Ranker().rank(df)

In [15]:
df.head()

,entity_id,year,operator_id,type,size,age,km,fuel,co2,fuel_per_km,co2_per_km,size_bucket,age_bucket,co2_per_km_lag1,cluster,percentile_co2,zscore_co2,rank_in_type,rank_global
0,0,2018,C-015,long-haul,21.680127,4,91468.944340,35.421193,93.511950,0.000387,0.001022,0,1,NaN,0,0.500000,-0.772947,61.0,213.0
1,0,2019,C-015,long-haul,21.680127,5,153835.212095,66.257263,174.919175,0.000431,0.001137,0,1,0.001022,0,0.916667,0.789709,74.0,269.0
2,0,2020,C-015,long-haul,21.680127,6,149495.965275,65.903525,173.985307,0.000441,0.001164,0,1,0.001137,4,0.933333,0.849900,64.0,241.0
3,0,2021,C-015,long-haul,21.680127,7,176381.939246,87.909431,232.080898,0.000498,0.001316,0,1,0.001164,4,1.000000,1.841436,85.0,333.0
4,0,2022,C-015,long-haul,21.680127,8,151980.920927,80.235483,211.821676,0.000528,0.001394,0,2,0.001316,4,0.833333,0.947931,90.0,347.0
